In [39]:
import numpy as np
import struct

In [40]:
class FP32:
    def __init__(self, value=0.0):
        """
        Initialize an FP32 object from a floating-point number.

        Args:
            value (float): The floating-point number.
        """
        self.value = value
        self.binary = self.float_to_binary(value)
        self.bias = 2 ** (8 - 1) - 1
        self.sign, self.exponent, self.mantissa, self.significand = self.split_fp32(self.binary)
        self.nonbiased_exponent = self.exponent - self.bias
    
    @staticmethod
    def float_to_binary(value):
        """
        Convert a floating-point number to its binary representation.

        Args:
            value (float): The floating-point number.

        Returns:
            str: The binary representation of the floating-point number.
        """
        [d] = struct.unpack(">L", struct.pack(">f", value))
        return f"{d:032b}"
    
    @staticmethod
    def split_fp32(binary):
        """
        Split the binary representation of an FP32 number into sign, exponent, and significand.

        Args:
            binary (str): The binary representation of the FP32 number.

        Returns:
            tuple: The sign, exponent, and significand bits.
        """
        sign = int(binary[0], 2)
        exponent = int(binary[1:9], 2)
        mantissa = int(binary[9:], 2)
        if exponent == 0xFF: # Special values
            if mantissa == 0:
                return sign, float('inf'), 0, 0
            else:
                return sign, float('nan'), 0, 0
        elif exponent == 0: # Denormalized number
            exponent = 1
            significand = mantissa
        else: # Normalized number
            significand = int('1' + binary[9:], 2)
        return sign, exponent, mantissa, significand
    
    @staticmethod
    def binary_to_float(binary):
        """
        Convert a binary representation to a floating-point number.

        Args:
            binary (str): The binary representation.

        Returns:
            float: The floating-point number.
        """
        int_rep = int(binary, 2)
        packed = struct.pack('>I', int_rep)
        return struct.unpack('>f', packed)[0]
    
    def __repr__(self):
        binary_exponent = f"{self.exponent:08b}"
        binary_significand = f"{self.significand:024b}"
        return (f"FP32(value={self.value}, sign={self.sign}, "
                f"nonbiased exponent={self.nonbiased_exponent} (binary={binary_exponent}), "
                f"significand binary={binary_significand})")

In [41]:
# Test the FP32 class
fp = FP32(2.5125)
print(fp)
print(fp.binary)
print(FP32.binary_to_float(fp.binary))
fp2 = FP32(FP32.binary_to_float(fp.binary))
print(fp2.binary)

FP32(value=2.5125, sign=0, nonbiased exponent=1 (binary=10000000), significand binary=101000001100110011001101)
01000000001000001100110011001101
2.512500047683716
01000000001000001100110011001101


In [86]:
class FP9:
    def __init__(self, value=0.0):
        """
        Initialize an FP9 object from a floating-point number.

        Args:
            value (float): The floating-point number.
        """
        self.value = value
        
        self.exponent_bits = 5
        self.bias = 2 ** (self.exponent_bits - 1) - 1
        self.mantissa_bits = 3
        precision_bits = self.mantissa_bits + 1
        self.exponent_min = -self.bias + 1
        self.exponent_max = self.bias
        self.min_value = 2 ** (self.exponent_min + 1 - precision_bits)
        self.min_normal = 2 ** self.exponent_min
        self.max_value = 2 ** self.exponent_max * (2 - 2 ** -self.mantissa_bits)

        self.binary = self.float_to_binary()
        self.sign, self.exponent, self.nonbiased_exponent, self.mantissa, self.significand = self.split_fp9()

    def float_to_binary(self):
        """
        Convert a floating-point number to its 9-bit binary representation.

        Args:
            value (float): The floating-point number.

        Returns:
            str: The 9-bit binary representation of the floating-point number.
        """
        if self.value == 0:
            return '000000000'  # Zero representation in FP9

        sign = 0 if self.value >= 0 else 1
        value = abs(self.value)

        if value < self.min_value:
            raise ValueError(f"Value {value} is too small for FP9 format.")
        if value > self.max_value:
            raise ValueError(f"Value {value} is too large for FP9 format.")
        
        self.is_subnormal = False
        if value < self.min_normal: # Denormalized number
            print(f"Value {value} is a denormalized number.")
            self.is_subnormal = True
        
        if self.is_subnormal:
            exponent = 0
            value = value / (2 ** self.exponent_min)
            mantissa = int(value * (1 << self.mantissa_bits))
        else:
            exponent = self.bias
            while value >= 2:
                value /= 2
                exponent += 1
            while value < 1:
                value *= 2
                exponent -= 1
            mantissa = int((value - 1) * (1 << self.mantissa_bits))

        return f"{sign:01b}{exponent:05b}{mantissa:03b}"
    
    def split_fp9(self):
        """
        Split the binary representation of an FP9 number into sign, exponent, and mantissa.

        Args:
            binary (str): The 9-bit binary representation of the FP9 number.

        Returns:
            tuple: The sign, exponent, and mantissa bits.
        """
        sign = int(self.binary[0], 2)  
        exponent = int(self.binary[1:6], 2)
        mantissa = int(self.binary[6:], 2)
        if exponent == 0:
            nonbiased_exponent = self.exponent_min
            significand = mantissa
        else:
            nonbiased_exponent = exponent - self.bias
            significand = 1 << 3 | mantissa
        return sign, exponent, nonbiased_exponent, mantissa, significand
    
    def binary_to_float(self):
        """
        Convert a 9-bit binary representation to a floating-point number.

        Args:
            binary (str): The 9-bit binary representation.

        Returns:
            float: The floating-point number.
        """

        value = (self.significand / 2 ** self.mantissa_bits) * 2 ** self.nonbiased_exponent
        return -value if self.sign == 1 else value
    
    def __repr__(self):
        binary_exponent = f"{self.exponent:05b}"
        binary_mantissa = f"{self.mantissa:03b}"
        binary_significand = f"{self.significand:04b}"
        return (f"FP9(value={self.value}, sign={self.sign}, "
                f"exponent={self.exponent} (binary={binary_exponent}), "
                f"mantissa={self.mantissa} (binary={binary_mantissa})), "
                f"significand={self.significand} (binary={binary_significand}), "
                f"nonbiased exponent={self.nonbiased_exponent},")

In [89]:
# Example Usage
def test_fp9_repr(float_value=3.5):
    fp9_value = FP9(float_value)
    print(fp9_value)
    print(f"Binary value: {fp9_value.binary}")
    print(f"Float value: {fp9_value.binary_to_float()}")

In [90]:
# Test the FP9 class
test_fp9_repr(3.5)
test_fp9_repr(-3.5)
test_fp9_repr(2.0)

FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
Binary value: 010000110
Float value: 3.5
FP9(value=-3.5, sign=1, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
Binary value: 110000110
Float value: -3.5
FP9(value=2.0, sign=0, exponent=16 (binary=10000), mantissa=0 (binary=000)), significand=8 (binary=1000), nonbiased exponent=1,
Binary value: 010000000
Float value: 2.0


In [45]:
def int_to_binary(num, bits):
    if num >= 0:
        binary = bin(num)[2:]  # Convert to binary and remove the '0b' prefix
        return binary.zfill(bits)  # Pad the binary number with leading zeros
    else:
        binary = bin(num & (2**bits - 1))[2:]  # Compute two's complement
        return binary.zfill(bits)  # Pad the binary number with leading zeros

In [91]:
def multiply_fp9(fp9_a, fp9_b, anchor=34):
    """
    Multiply two FP9 values using their mantissa and exponent bits.

    Args:
        fp9_a (FP9): The first FP9 value.
        fp9_b (FP9): The second FP9 value.

    Returns:             
        FP9: The result of the multiplication as an FP9 value.
    """
    # Extract sign, exponent, and mantissa
    sign_a, exponent_a, significand_a = fp9_a.sign, fp9_a.exponent, fp9_a.significand
    sign_b, exponent_b, significand_b = fp9_b.sign, fp9_b.exponent, fp9_b.significand
    
    # Calculate the resulting sign
    sign_result = sign_a ^ sign_b

    # Multiply the significands (9b)
    significand_mul = significand_a * significand_b
    if sign_result == 1:
        significand_mul = -significand_mul
    
    # Adjust the resulting exponent (excess-31 notation)
    exponent_e31_sum = exponent_a + exponent_b + 1
    
    # Right shift the significand by anchor point - exponent
    # sum of four 9-bit numbers can be at most 11 bits, for 69 bits output we need to shift by 69 - 11 = 58
    # 58-30=28 plus inherit 6 fractional bits from the multiplication -> point moves to 28+6=34
    significand_result = (significand_mul << 58) >> (anchor - (exponent_e31_sum - 31) - 4)
    
    print(f"exponent_e31_sum: {exponent_e31_sum}")
    print(f"significand_mul: {significand_mul}")
    print(f"significand_mul in binary: {int_to_binary(significand_mul, 9)}")
    print(f"significand_result in binary: {int_to_binary(significand_result, 69)}")

    return significand_result

In [94]:
# Example Usage
def test_multiply_fp9(a, b):
    fp9_a = FP9(a)
    fp9_b = FP9(b)
    print(f"Multiplying {fp9_a.binary_to_float()} and {fp9_b.binary_to_float()}")
    # Print binary a and b
    print(f"a: {fp9_a}")
    print(f"b: {fp9_b}")
    result = multiply_fp9(fp9_a, fp9_b)

In [95]:
test_multiply_fp9(1.5, 2.5)

Multiplying 1.5 and 2.5
a: FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
b: FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
exponent_e31_sum: 32
significand_mul: 120
significand_mul in binary: 001111000
significand_result in binary: 000000000000000000000000000000000111100000000000000000000000000000000


In [96]:
test_multiply_fp9(3.5, 2.0)

Multiplying 3.5 and 2.0
a: FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
b: FP9(value=2.0, sign=0, exponent=16 (binary=10000), mantissa=0 (binary=000)), significand=8 (binary=1000), nonbiased exponent=1,
exponent_e31_sum: 33
significand_mul: 112
significand_mul in binary: 001110000
significand_result in binary: 000000000000000000000000000000001110000000000000000000000000000000000


In [97]:
test_multiply_fp9(3.5, -2.0)

Multiplying 3.5 and -2.0
a: FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
b: FP9(value=-2.0, sign=1, exponent=16 (binary=10000), mantissa=0 (binary=000)), significand=8 (binary=1000), nonbiased exponent=1,
exponent_e31_sum: 33
significand_mul: -112
significand_mul in binary: 110010000
significand_result in binary: 111111111111111111111111111111110010000000000000000000000000000000000


In [98]:
def sum_fp9_fp32(fp32, fp9_significand_result, fp9_scale):
    # Original point is at 23, needs to be up shifted to 34 (34-23)
    scaled_anchor = 34 - fp9_scale
    shift_fp32_acc = scaled_anchor - 23 + fp32.nonbiased_exponent
    print(f"shift_fp32_acc: {shift_fp32_acc}")
    if fp32.sign == 1:
        fp32_significand_result = -fp32.significand
    else:
        fp32_significand_result = fp32.significand
    # -1 for the sign bit
    MAX_SHIFT = 94 - 24 - 1
    if shift_fp32_acc > MAX_SHIFT:
        raise ValueError(f"Shift amount {shift_fp32_acc} exceeds maximum shift amount of {MAX_SHIFT}")
    elif shift_fp32_acc > 0:
        shifted_significant_result = fp32_significand_result << shift_fp32_acc
    else:
        shifted_significant_result = fp32_significand_result >> -shift_fp32_acc
    print(f"FP9 multiplication significand result:\n{int_to_binary(fp9_significand_result, 94)}")
    # print(f"Shifted FP9 multiplication significand result:\n{fp9_significand_result:094b}")
    print(f"Shifted FP32 significand result:\n{int_to_binary(shifted_significant_result, 94)}")
    # print(f"Shifted FP32 significand result:\n{shifted_significant_result:094b}")
    sum_significant = shifted_significant_result + fp9_significand_result
    return sum_significant

In [99]:
# Sum FP9 multiplication result with an FP32 value
def test_fp9_fp32_sum(a, b, s, scale=0):
    fp32 = FP32(s)
    fp9_a = FP9(a)
    fp9_b = FP9(b)
    print(f"Computing {fp9_a.value} * {fp9_b.value} * 2 ^ {scale} + {fp32.value}")
    print(f"FP9 (a): {fp9_a}")
    print(f"FP9 (b): {fp9_b}")
    print(f"FP32(s): {fp32}")
    result = multiply_fp9(fp9_a, fp9_b)
    sum_result = sum_fp9_fp32(fp32, result, scale)
    # Compute the scaled anchor point
    scaled_anchor = 34 - scale
    if scaled_anchor < 0:
        # All bits are integer bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    else:
        # Find the upper 94-34=60 bits of the result
        upper_60_bits = sum_result >> scaled_anchor
        # print(f"Upper 60 bits of the result in binary:\n{upper_60_bits:060b}, decimal: {upper_60_bits}")
        print(f"Integer {94-scaled_anchor} bits of the result in binary:\n{int_to_binary(upper_60_bits, 94-scaled_anchor)}, decimal: {upper_60_bits}")
        # Find lower 34 bits of the result
        lower_34_bits = sum_result & ((1 << scaled_anchor) - 1)
        # print(f"Lower 34 bits of the result in binary:\n{lower_34_bits:034b}, decimal: {lower_34_bits/2**34}")
        print(f"Fractional {scaled_anchor} bits of the result in binary:\n{int_to_binary(lower_34_bits, scaled_anchor)}, decimal: {lower_34_bits/2**scaled_anchor}")
        print(f"Result in binary:\n{int_to_binary(sum_result, 94)}")
    decimal_result = sum_result / 2**scaled_anchor
    # Print in decimal
    print(f"Decimal result: {decimal_result:.34f}")
    # Compare the result with the actual value
    actual_result = a * b * 2 ** scale + s
    print(f"Actual result:  {actual_result:.34f}")

In [100]:
test_fp9_fp32_sum(1.5, 2.5, 2.5)

Computing 1.5 * 2.5 * 2 ^ 0 + 2.5
FP9 (a): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=2.5, sign=0, nonbiased exponent=1 (binary=10000000), significand binary=101000000000000000000000)
exponent_e31_sum: 32
significand_mul: 120
significand_mul in binary: 001111000
significand_result in binary: 000000000000000000000000000000000111100000000000000000000000000000000
shift_fp32_acc: 12
FP9 multiplication significand result:
0000000000000000000000000000000000000000000000000000000000111100000000000000000000000000000000
Shifted FP32 significand result:
0000000000000000000000000000000000000000000000000000000000101000000000000000000000000000000000
Integer 60 bits of the result in binary:
000000000000000000000000000000000000000000000000000000000110, decimal: 

In [101]:
test_fp9_fp32_sum(30720, 2.5, 8.5)

Computing 30720 * 2.5 * 2 ^ 0 + 8.5
FP9 (a): FP9(value=30720, sign=0, exponent=29 (binary=11101), mantissa=7 (binary=111)), significand=15 (binary=1111), nonbiased exponent=14,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=8.5, sign=0, nonbiased exponent=3 (binary=10000010), significand binary=100010000000000000000000)
exponent_e31_sum: 46
significand_mul: 150
significand_mul in binary: 010010110
significand_result in binary: 000000000000000000100101100000000000000000000000000000000000000000000
shift_fp32_acc: 14
FP9 multiplication significand result:
0000000000000000000000000000000000000000000100101100000000000000000000000000000000000000000000
Shifted FP32 significand result:
0000000000000000000000000000000000000000000000000000000010001000000000000000000000000000000000
Integer 60 bits of the result in binary:
000000000000000000000000000000000000000000010010110000001000, deci

In [102]:
test_fp9_fp32_sum(61440, 61440, 8.5)

Computing 61440 * 61440 * 2 ^ 0 + 8.5
FP9 (a): FP9(value=61440, sign=0, exponent=30 (binary=11110), mantissa=7 (binary=111)), significand=15 (binary=1111), nonbiased exponent=15,
FP9 (b): FP9(value=61440, sign=0, exponent=30 (binary=11110), mantissa=7 (binary=111)), significand=15 (binary=1111), nonbiased exponent=15,
FP32(s): FP32(value=8.5, sign=0, nonbiased exponent=3 (binary=10000010), significand binary=100010000000000000000000)
exponent_e31_sum: 61
significand_mul: 225
significand_mul in binary: 011100001
significand_result in binary: 000111000010000000000000000000000000000000000000000000000000000000000
shift_fp32_acc: 14
FP9 multiplication significand result:
0000000000000000000000000000111000010000000000000000000000000000000000000000000000000000000000
Shifted FP32 significand result:
0000000000000000000000000000000000000000000000000000000010001000000000000000000000000000000000
Integer 60 bits of the result in binary:
000000000000000000000000000011100001000000000000000000001000,

In [103]:
# Max FP32 exponent to stay within 94b is 94-1-24-11 = 58
test_fp9_fp32_sum(61440, 61440, 2**58*(2-2**-23))

Computing 61440 * 61440 * 2 ^ 0 + 5.764607179436851e+17
FP9 (a): FP9(value=61440, sign=0, exponent=30 (binary=11110), mantissa=7 (binary=111)), significand=15 (binary=1111), nonbiased exponent=15,
FP9 (b): FP9(value=61440, sign=0, exponent=30 (binary=11110), mantissa=7 (binary=111)), significand=15 (binary=1111), nonbiased exponent=15,
FP32(s): FP32(value=5.764607179436851e+17, sign=0, nonbiased exponent=58 (binary=10111001), significand binary=111111111111111111111111)
exponent_e31_sum: 61
significand_mul: 225
significand_mul in binary: 011100001
significand_result in binary: 000111000010000000000000000000000000000000000000000000000000000000000
shift_fp32_acc: 69
FP9 multiplication significand result:
0000000000000000000000000000111000010000000000000000000000000000000000000000000000000000000000
Shifted FP32 significand result:
0111111111111111111111111000000000000000000000000000000000000000000000000000000000000000000000
Integer 60 bits of the result in binary:
011111111111111111111111

In [104]:
test_fp9_fp32_sum(-1.5, 2.5, 2.5)

Computing -1.5 * 2.5 * 2 ^ 0 + 2.5
FP9 (a): FP9(value=-1.5, sign=1, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=2.5, sign=0, nonbiased exponent=1 (binary=10000000), significand binary=101000000000000000000000)
exponent_e31_sum: 32
significand_mul: -120
significand_mul in binary: 110001000
significand_result in binary: 111111111111111111111111111111111000100000000000000000000000000000000
shift_fp32_acc: 12
FP9 multiplication significand result:
1111111111111111111111111111111111111111111111111111111111000100000000000000000000000000000000
Shifted FP32 significand result:
0000000000000000000000000000000000000000000000000000000000101000000000000000000000000000000000
Integer 60 bits of the result in binary:
111111111111111111111111111111111111111111111111111111111110, decima

In [105]:
int_to_binary(-120, 9)

'110001000'

In [106]:
test_fp9_fp32_sum(1.5, 2.5, -2.5)

Computing 1.5 * 2.5 * 2 ^ 0 + -2.5
FP9 (a): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=-2.5, sign=1, nonbiased exponent=1 (binary=10000000), significand binary=101000000000000000000000)
exponent_e31_sum: 32
significand_mul: 120
significand_mul in binary: 001111000
significand_result in binary: 000000000000000000000000000000000111100000000000000000000000000000000
shift_fp32_acc: 12
FP9 multiplication significand result:
0000000000000000000000000000000000000000000000000000000000111100000000000000000000000000000000
Shifted FP32 significand result:
1111111111111111111111111111111111111111111111111111111111011000000000000000000000000000000000
Integer 60 bits of the result in binary:
000000000000000000000000000000000000000000000000000000000001, decimal

In [107]:
test_fp9_fp32_sum(1.5, 2.5, -2.5, 1)

Computing 1.5 * 2.5 * 2 ^ 1 + -2.5
FP9 (a): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=-2.5, sign=1, nonbiased exponent=1 (binary=10000000), significand binary=101000000000000000000000)
exponent_e31_sum: 32
significand_mul: 120
significand_mul in binary: 001111000
significand_result in binary: 000000000000000000000000000000000111100000000000000000000000000000000
shift_fp32_acc: 11
FP9 multiplication significand result:
0000000000000000000000000000000000000000000000000000000000111100000000000000000000000000000000
Shifted FP32 significand result:
1111111111111111111111111111111111111111111111111111111111101100000000000000000000000000000000
Integer 61 bits of the result in binary:
0000000000000000000000000000000000000000000000000000000000101, decima

In [108]:
fp9_a_value = 1.5
fp9_b_value = 2.5
fp32_s_value = -2.5*2**-32
fp9_7b_scale = 1
test_fp9_fp32_sum(fp9_a_value, fp9_b_value, fp32_s_value, fp9_7b_scale)

Computing 1.5 * 2.5 * 2 ^ 1 + -5.820766091346741e-10
FP9 (a): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=-5.820766091346741e-10, sign=1, nonbiased exponent=-31 (binary=01100000), significand binary=101000000000000000000000)
exponent_e31_sum: 32
significand_mul: 120
significand_mul in binary: 001111000
significand_result in binary: 000000000000000000000000000000000111100000000000000000000000000000000
shift_fp32_acc: -21
FP9 multiplication significand result:
0000000000000000000000000000000000000000000000000000000000111100000000000000000000000000000000
Shifted FP32 significand result:
1111111111111111111111111111111111111111111111111111111111111111111111111111111111111111111011
Integer 61 bits of the result in binary:
000000000000000000000000000000

In [109]:
fp9_a_value = 1.5
fp9_b_value = 2.5
fp32_s_value = -2.5*2**62
fp9_7b_scale = 63
test_fp9_fp32_sum(fp9_a_value, fp9_b_value, fp32_s_value, fp9_7b_scale)

Computing 1.5 * 2.5 * 2 ^ 63 + -1.152921504606847e+19
FP9 (a): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP32(s): FP32(value=-1.152921504606847e+19, sign=1, nonbiased exponent=63 (binary=10111110), significand binary=101000000000000000000000)
exponent_e31_sum: 32
significand_mul: 120
significand_mul in binary: 001111000
significand_result in binary: 000000000000000000000000000000000111100000000000000000000000000000000
shift_fp32_acc: 11
FP9 multiplication significand result:
0000000000000000000000000000000000000000000000000000000000111100000000000000000000000000000000
Shifted FP32 significand result:
1111111111111111111111111111111111111111111111111111111111101100000000000000000000000000000000
Result in binary with a scale of 29:
00000000000000000000000000000000000

In [110]:
# Create a vector of 9-bit floating-point numbers
def create_fp9_vector(values):
    return [FP9(value) for value in values]

# Multiply two vectors of FP9 values
def multiply_fp9_vectors(fp9_vector_a, fp9_vector_b):
    if len(fp9_vector_a) != len(fp9_vector_b):
        raise ValueError("Vectors must have the same length")
    return [multiply_fp9(fp9_a, fp9_b) for fp9_a, fp9_b in zip(fp9_vector_a, fp9_vector_b)]


In [111]:
def sum_fp9_sop_fp32(a, b, s, scale=0):
    fp32 = FP32(s)
    fp9_vector_a = create_fp9_vector(a)
    fp9_vector_b = create_fp9_vector(b)
    print(f"Computing SOP of two vectors of FP9 values and summing with an FP32 value")
    # Print the FP9 vectors each in a new line
    for i, (fp9_a, fp9_b) in enumerate(zip(fp9_vector_a, fp9_vector_b)):
        print(f"FP9 (a[{i}]): {fp9_a}")
        print(f"FP9 (b[{i}]): {fp9_b}")
    print(f"FP32(s): {fp32}")

    products_fp9_vectors = multiply_fp9_vectors(fp9_vector_a, fp9_vector_b)
    # Sum the products of the FP9 vectors
    sum_fp9_vectors = sum(products_fp9_vectors)
    sum_result = sum_fp9_fp32(fp32, sum_fp9_vectors, scale)

    # Compute the scaled anchor point
    scaled_anchor = 34 - scale
    if scaled_anchor < 0:
        # All bits are integer bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    elif scaled_anchor >= 94:
        # All bits are fractional bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    else:
        # Find the upper 94-34=60 bits of the result
        integer_bits = sum_result >> scaled_anchor
        print(f"Integer {94-scaled_anchor} bits of the result in binary:\n{int_to_binary(integer_bits, 94-scaled_anchor)}, decimal: {integer_bits}")
        # Find lower 34 bits of the result
        integer_bits = sum_result & ((1 << scaled_anchor) - 1)
        print(f"Fractional {scaled_anchor} bits of the result in binary:\n{int_to_binary(integer_bits, scaled_anchor)}, decimal: {integer_bits/2**scaled_anchor}")
        print(f"Result in binary:\n{int_to_binary(sum_result, 94)}")
    decimal_result = sum_result / 2**scaled_anchor
    # Print in decimal
    print(f"Decimal result: {decimal_result:.50f}")
    # Compare the result with the actual value
    actual_result = sum([a * b for a, b in zip(a, b)]) * 2 ** scale + s
    print(f"Actual result:  {actual_result:.50f}")

    if decimal_result != actual_result:
        raise ValueError("The result does not match the actual value")

In [112]:
# Example Usage
a = [1.5, 2.5, 3.5, 4.5]
b = [2.5, 3.5, 4.5, 5.5]
s = 5.25
scale = 1
test_fp9_sop_fp32_sum(a, b, s, scale)

Computing SOP of two vectors of FP9 values and summing with an FP32 value
FP9 (a[0]): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b[0]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (a[1]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (b[1]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (a[2]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (b[2]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary=001)), significand=9 (binary=1001), nonbiased exponent=2,
FP9 (a[3]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary

In [113]:
a = [1.5, 2.5, 3.5, 4.5]
a = [x * 2**13 for x in a]
b = [2.5, 3.5, 4.5, 5.5]
b = [x * 2**13 for x in b]
s = -5.25 * 2**57
scale = 1
test_fp9_sop_fp32_sum(a, b, s, scale)

Computing SOP of two vectors of FP9 values and summing with an FP32 value
FP9 (a[0]): FP9(value=12288.0, sign=0, exponent=28 (binary=11100), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=13,
FP9 (b[0]): FP9(value=20480.0, sign=0, exponent=29 (binary=11101), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=14,
FP9 (a[1]): FP9(value=20480.0, sign=0, exponent=29 (binary=11101), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=14,
FP9 (b[1]): FP9(value=28672.0, sign=0, exponent=29 (binary=11101), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=14,
FP9 (a[2]): FP9(value=28672.0, sign=0, exponent=29 (binary=11101), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=14,
FP9 (b[2]): FP9(value=36864.0, sign=0, exponent=30 (binary=11110), mantissa=1 (binary=001)), significand=9 (binary=1001), nonbiased exponent=15,
FP9 (a[3]): FP9(value=36864.0, sign=0, exponent=30 

In [114]:
a = [1.5, 2.5, 3.5, 4.5]
b = [2.5, 3.5, 4.5, 5.5]
s = -525
scale = 1
test_fp9_sop_fp32_sum(a, b, s, scale)

Computing SOP of two vectors of FP9 values and summing with an FP32 value
FP9 (a[0]): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b[0]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (a[1]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (b[1]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (a[2]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (b[2]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary=001)), significand=9 (binary=1001), nonbiased exponent=2,
FP9 (a[3]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary

In [115]:
a = [1.5, 2.5, 3.5, 4.5]
b = [2.5, 3.5, 4.5, 5.5]
s = 2**-126*0.25
scale = -128
test_fp9_sop_fp32_sum(a, b, s, scale)

Computing SOP of two vectors of FP9 values and summing with an FP32 value
FP9 (a[0]): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b[0]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (a[1]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (b[1]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (a[2]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (b[2]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary=001)), significand=9 (binary=1001), nonbiased exponent=2,
FP9 (a[3]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary

In [116]:
def sum_fp9_sop_fp32(a, b, s, scale=0):
    fp32 = FP32(s)
    fp9_vector_a = create_fp9_vector(a)
    fp9_vector_b = create_fp9_vector(b)
    print(f"Computing SOP of two vectors of FP9 values and summing with an FP32 value")
    # Print the FP9 vectors each in a new line
    for i, (fp9_a, fp9_b) in enumerate(zip(fp9_vector_a, fp9_vector_b)):
        print(f"FP9 (a[{i}]): {fp9_a}")
        print(f"FP9 (b[{i}]): {fp9_b}")
    print(f"FP32(s): {fp32}")

    products_fp9_vectors = multiply_fp9_vectors(fp9_vector_a, fp9_vector_b)
    # Sum the products of the FP9 vectors
    sum_fp9_vectors = sum(products_fp9_vectors)
    sum_result = sum_fp9_fp32(fp32, sum_fp9_vectors, scale)

    # Compute the scaled anchor point
    scaled_anchor = 34 - scale
    if scaled_anchor < 0:
        # All bits are integer bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    elif scaled_anchor >= 94:
        # All bits are fractional bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    else:
        # Find the upper 94-34=60 bits of the result
        integer_bits = sum_result >> scaled_anchor
        print(f"Integer {94-scaled_anchor} bits of the result in binary:\n{int_to_binary(integer_bits, 94-scaled_anchor)}, decimal: {integer_bits}")
        # Find lower 34 bits of the result
        integer_bits = sum_result & ((1 << scaled_anchor) - 1)
        print(f"Fractional {scaled_anchor} bits of the result in binary:\n{int_to_binary(integer_bits, scaled_anchor)}, decimal: {integer_bits/2**scaled_anchor}")
        print(f"Result in binary:\n{int_to_binary(sum_result, 94)}")
    decimal_result = sum_result / 2**scaled_anchor
    # Print in decimal
    print(f"Decimal result: {decimal_result:.50f}")
    # Compare the result with the actual value
    actual_result = sum([a * b for a, b in zip(a, b)]) * 2 ** scale + s
    print(f"Actual result:  {actual_result:.50f}")

    if decimal_result != actual_result:
        raise ValueError("The result does not match the actual value")

In [119]:
a = [1.5, 2.5, 3.5, 4.5]
b = [2.5, 3.5, 4.5, 5.5]
s = -525
scale = 1
sum_fp9_sop_fp32(a, b, s, scale)

Computing SOP of two vectors of FP9 values and summing with an FP32 value
FP9 (a[0]): FP9(value=1.5, sign=0, exponent=15 (binary=01111), mantissa=4 (binary=100)), significand=12 (binary=1100), nonbiased exponent=0,
FP9 (b[0]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (a[1]): FP9(value=2.5, sign=0, exponent=16 (binary=10000), mantissa=2 (binary=010)), significand=10 (binary=1010), nonbiased exponent=1,
FP9 (b[1]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (a[2]): FP9(value=3.5, sign=0, exponent=16 (binary=10000), mantissa=6 (binary=110)), significand=14 (binary=1110), nonbiased exponent=1,
FP9 (b[2]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary=001)), significand=9 (binary=1001), nonbiased exponent=2,
FP9 (a[3]): FP9(value=4.5, sign=0, exponent=17 (binary=10001), mantissa=1 (binary

Backup

In [ ]:
# def multiply_fp9(fp9_a, fp9_b, anchor=34):
#     """
#     Multiply two FP9 values using their mantissa and exponent bits.

#     Args:
#         fp9_a (FP9): The first FP9 value.
#         fp9_b (FP9): The second FP9 value.

#     Returns:             
#         FP9: The result of the multiplication as an FP9 value.
#     """
#     # Extract sign, exponent, and mantissa
#     sign_a, exponent_a, significand_a = fp9_a.sign, fp9_a.exponent, fp9_a.significand
#     sign_b, exponent_b, significand_b = fp9_b.sign, fp9_b.exponent, fp9_b.significand
    
#     # Calculate the resulting sign
#     sign_result = sign_a ^ sign_b

#     # Multiply the significands (9b)
#     significand_mul = significand_a * significand_b
#     if sign_result == 1:
#         significand_mul = -significand_mul
    
#     # Adjust the resulting exponent (excess-31 notation)
#     exponent_e31_sum = exponent_a + exponent_b + 1
    
#     # Right shift the significand by anchor point - exponent
#     # sum of four 9-bit numbers can be at most 11 bits, for 69 bits output we need to shift by 69 - 11 = 58
#     # 58-30=28 plus inherit 6 fractional bits from the multiplication -> point moves to 28+6=34
#     significand_result = (significand_mul << 58) >> (anchor - (exponent_e31_sum - 31) - 4)
    
#     print(f"exponent_e31_sum: {exponent_e31_sum}")
#     print(f"significand_mul: {significand_mul}")
#     print(f"significand_mul in binary: {int_to_binary(significand_mul, 9)}")
#     print(f"significand_result in binary: {int_to_binary(significand_result, 69)}")

#     return significand_result

In [ ]:
# class FP9:
#     def __init__(self, value=0.0):
#         """
#         Initialize an FP9 object from a floating-point number.

#         Args:
#             value (float): The floating-point number.
#         """
#         self.value = value
        
#         self.exponent_bits = 5
#         self.bias = 2 ** (self.exponent_bits - 1) - 1
#         self.mantissa_bits = 3
#         precision_bits = self.mantissa_bits + 1
#         self.exponent_min = -self.bias + 1
#         self.exponent_max = self.bias
#         self.min_value = 2 ** (self.exponent_min + 1 - precision_bits)
#         self.min_normal = 2 ** self.exponent_min
#         self.max_value = 2 ** self.exponent_max * (2 - 2 ** -self.mantissa_bits)

#         self.binary = self.float_to_binary()
#         self.sign, self.exponent, self.nonbiased_exponent, self.mantissa, self.significand = self.split_fp9()

#     def float_to_binary(self):
#         """
#         Convert a floating-point number to its 9-bit binary representation.

#         Args:
#             value (float): The floating-point number.

#         Returns:
#             str: The 9-bit binary representation of the floating-point number.
#         """
#         if self.value == 0:
#             return '000000000'  # Zero representation in FP9

#         sign = 0 if self.value >= 0 else 1
#         value = abs(self.value)

#         if value < self.min_value:
#             raise ValueError(f"Value {value} is too small for FP9 format.")
#         if value > self.max_value:
#             raise ValueError(f"Value {value} is too large for FP9 format.")
        
#         self.is_subnormal = False
#         if value < self.min_normal: # Denormalized number
#             print(f"Value {value} is a denormalized number.")
#             self.is_subnormal = True
        
#         if self.is_subnormal:
#             exponent = 0
#             value = value / (2 ** self.exponent_min)
#             mantissa = int(value * (1 << self.mantissa_bits))
#         else:
#             exponent = self.bias
#             while value >= 2:
#                 value /= 2
#                 exponent += 1
#             while value < 1:
#                 value *= 2
#                 exponent -= 1
#             mantissa = int((value - 1) * (1 << self.mantissa_bits))

#         return f"{sign:01b}{exponent:05b}{mantissa:03b}"
    
#     def split_fp9(self):
#         """
#         Split the binary representation of an FP9 number into sign, exponent, and mantissa.

#         Args:
#             binary (str): The 9-bit binary representation of the FP9 number.

#         Returns:
#             tuple: The sign, exponent, and mantissa bits.
#         """
#         sign = int(self.binary[0], 2)  
#         exponent = int(self.binary[1:6], 2)
#         mantissa = int(self.binary[6:], 2)
#         if exponent == 0:
#             nonbiased_exponent = self.exponent_min
#             significand = mantissa
#         else:
#             nonbiased_exponent = exponent - self.bias
#             significand = 1 << 3 | mantissa
#         return sign, exponent, nonbiased_exponent, mantissa, significand

#     @staticmethod
#     def binary_to_float(binary):
#         """
#         Convert a 9-bit binary representation to a floating-point number.

#         Args:
#             binary (str): The 9-bit binary representation.

#         Returns:
#             float: The floating-point number.
#         """

#         sign = int(binary[0], 2)
#         exponent = int(binary[1:6], 2)
#         mantissa = int(binary[6:], 2) / (1 << 3)

#         if exponent == 31: # Inf or NaN
#             raise ValueError("Special values are not yet supported")
#         elif exponent == 0:
#             exponent = -14
#             mantissa = mantissa
#         else:
#             exponent = exponent - 15
#             mantissa = 1 + mantissa
        
#         value = mantissa * (2 ** exponent)
#         return -value if sign == 1 else value
        
#     def __repr__(self):
#         binary_exponent = f"{self.exponent:05b}"
#         binary_mantissa = f"{self.mantissa:03b}"
#         binary_significand = f"{self.significand:04b}"
#         return (f"FP9(value={self.value}, sign={self.sign}, "
#                 f"exponent={self.exponent} (binary={binary_exponent}), "
#                 f"mantissa={self.mantissa} (binary={binary_mantissa})), "
#                 f"significand={self.significand} (binary={binary_significand}), "
#                 f"nonbiased exponent={self.nonbiased_exponent})")

#     @classmethod
#     def generate_random(cls, seed=None, allow_special_values=False):
#         """
#         Generate a random FP9 value.

#         Args:
#             seed (int, optional): A seed for the random number generator. Defaults to None.
#             allow_special_values (bool, optional): If True, exponent can be up to 31 (special values).
#                                                    If False, exponent is limited to 30. Defaults to False.

#         Returns:
#             FP9: A randomly generated FP9 value.
#         """
#         if seed is not None:
#             random.seed(seed)
#         else:
#             random.seed()
        
#         # Randomly select the sign bit (0 or 1)
#         sign = random.randint(0, 1)

#         # Set the maximum exponent depending on whether special values are allowed
#         max_exponent = 31 if allow_special_values else 30

#         # Randomly select the exponent (5 bits, range 0 to 31)
#         exponent = random.randint(0, max_exponent)
        
#         # Randomly select the mantissa (3 bits, range 0 to 7)
#         mantissa = random.randint(0, 7)
        
#         # Construct the FP9 binary representation
#         binary = f"{sign:01b}{exponent:05b}{mantissa:03b}"
        
#         # Convert the binary representation to a floating-point number
#         value = cls.binary_to_float(binary)
        
#         # Return the FP9 object
#         return cls(value)


In [ ]:
def sum_fp9_fp32(fp32, fp9_significand_result, fp9_scale):
    # Original point is at 23, needs to be up shifted to 34 (34-23)
    scaled_anchor = 34 - fp9_scale
    shift_fp32_acc = scaled_anchor - 23 + fp32.nonbiased_exponent
    print(f"shift_fp32_acc: {shift_fp32_acc}")
    if fp32.sign == 1:
        fp32_significand_result = -fp32.significand
    else:
        fp32_significand_result = fp32.significand
    # -1 for the sign bit
    MAX_SHIFT = 94 - 24 - 1
    if shift_fp32_acc > MAX_SHIFT:
        raise ValueError(f"Shift amount {shift_fp32_acc} exceeds maximum shift amount of {MAX_SHIFT}")
    elif shift_fp32_acc > 0:
        shifted_significant_result = fp32_significand_result << shift_fp32_acc
    else:
        shifted_significant_result = fp32_significand_result >> -shift_fp32_acc
    print(f"FP9 multiplication significand result:\n{int_to_binary(fp9_significand_result, 94)}")
    # print(f"Shifted FP9 multiplication significand result:\n{fp9_significand_result:094b}")
    print(f"Shifted FP32 significand result:\n{int_to_binary(shifted_significant_result, 94)}")
    # print(f"Shifted FP32 significand result:\n{shifted_significant_result:094b}")
    sum_significant = shifted_significant_result + fp9_significand_result
    return sum_significant

In [ ]:
def sum_fp9_sop_fp32(a, b, s, scale=0):
    fp32 = FP32(s)
    fp9_vector_a = create_fp9_vector(a)
    fp9_vector_b = create_fp9_vector(b)
    print(f"Computing SOP of two vectors of FP9 values and summing with an FP32 value")
    # Print the FP9 vectors each in a new line
    for i, (fp9_a, fp9_b) in enumerate(zip(fp9_vector_a, fp9_vector_b)):
        print(f"FP9 (a[{i}]): {fp9_a}")
        print(f"FP9 (b[{i}]): {fp9_b}")
    print(f"FP32(s): {fp32}")

    products_fp9_vectors = multiply_fp9_vectors(fp9_vector_a, fp9_vector_b)
    # Sum the products of the FP9 vectors
    sum_fp9_vectors = sum(products_fp9_vectors)
    sum_result = sum_fp9_fp32(fp32, sum_fp9_vectors, scale)

    # Compute the scaled anchor point
    scaled_anchor = 34 - scale
    if scaled_anchor < 0:
        # All bits are integer bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    elif scaled_anchor >= 94:
        # All bits are fractional bits
        print(f"Result in binary with a scale of {-scaled_anchor}:\n{int_to_binary(sum_result, 94)}")
    else:
        # Find the upper 94-34=60 bits of the result
        integer_bits = sum_result >> scaled_anchor
        print(f"Integer {94-scaled_anchor} bits of the result in binary:\n{int_to_binary(integer_bits, 94-scaled_anchor)}, decimal: {integer_bits}")
        # Find lower 34 bits of the result
        integer_bits = sum_result & ((1 << scaled_anchor) - 1)
        print(f"Fractional {scaled_anchor} bits of the result in binary:\n{int_to_binary(integer_bits, scaled_anchor)}, decimal: {integer_bits/2**scaled_anchor}")
        print(f"Result in binary:\n{int_to_binary(sum_result, 94)}")
    decimal_result = sum_result / 2**scaled_anchor
    # Print in decimal
    print(f"Decimal result: {decimal_result:.50f}")
    # Compare the result with the actual value
    actual_result = sum([a * b for a, b in zip(a, b)]) * 2 ** scale + s
    print(f"Actual result:  {actual_result:.50f}")

    if decimal_result != actual_result:
        raise ValueError("The result does not match the actual value")
    